# 🚦 Smart Traffic Annotation System

**AI-powered traffic video annotation with real-time vehicle detection, tracking & analytics**

| Feature | Technology |
|---|---|
| **Detection** | YOLOv8 (Ultralytics) |
| **Tracking** | DeepSORT |
| **Backend** | FastAPI + SQLite |
| **Frontend** | HTML5 Canvas + JS |
| **Analytics** | Counting, Speed, Lane Analysis, Safety (TTC) |
| **Export** | COCO, YOLO, VOC, CSV, JSON |

> **Run all cells in order. Enable GPU:** `Runtime > Change runtime type > T4 GPU`


## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Step 2: Install Dependencies

> **Wait for this cell to finish completely.** If it asks to restart the runtime, click **Restart session**, then **skip this cell** and continue from Step 3.

In [ ]:
# Install only what Colab doesn't already have
# DO NOT pin numpy/opencv versions - Colab has compatible ones pre-installed
!pip install -q fastapi uvicorn[standard] python-multipart
!pip install -q sqlalchemy pydantic python-dotenv
!pip install -q ultralytics deep-sort-realtime
!pip install -q pyngrok

# Verify
import ultralytics; print(f'ultralytics: {ultralytics.__version__}')
import fastapi; print(f'fastapi: {fastapi.__version__}')
import cv2; print(f'opencv: {cv2.__version__}')
import numpy; print(f'numpy: {numpy.__version__}')
from deep_sort_realtime.deepsort_tracker import DeepSort; print('deep-sort: OK')
from pyngrok import ngrok; print('pyngrok: OK')
print('\nAll packages ready!')


## Step 3: Clone Repository

In [ ]:
import os
!rm -rf /content/traffic-app
!git clone https://github.com/Dakshbumb/traffic-annotation-system.git /content/traffic-app
os.makedirs('/content/traffic-app/backend/uploads', exist_ok=True)
os.makedirs('/content/traffic-app/backend/exports', exist_ok=True)
os.chdir('/content/traffic-app/backend')
print(f'Working dir: {os.getcwd()}')
print('Ready!')


## Step 4: Download YOLOv8 Model

In [ ]:
import os
os.chdir('/content/traffic-app/backend')
from ultralytics import YOLO
print('Downloading YOLOv8m model...')
model = YOLO('yolov8m.pt')
print(f'Model ready! ({os.path.getsize("yolov8m.pt") / 1024 / 1024:.1f} MB)')


## Step 5: Setup ngrok

1. Sign up free at [ngrok.com](https://ngrok.com)
2. Copy authtoken from [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste below


In [ ]:
from pyngrok import ngrok
NGROK_TOKEN = input('Enter ngrok authtoken: ')
ngrok.set_auth_token(NGROK_TOKEN)
print('ngrok configured!')


## Step 6: Start Server

> Click the **Public URL** to open the web interface!

In [ ]:
import subprocess, time, os, torch
from pyngrok import ngrok

os.chdir('/content/traffic-app/backend')

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
print('Starting server...')
time.sleep(8)

if server.poll() is not None:
    print('Server failed to start!')
    print(server.stderr.read().decode())
else:
    public_url = ngrok.connect(8000)
    print('=' * 60)
    print('SMART TRAFFIC ANNOTATION SYSTEM')
    print('=' * 60)
    print(f'\nPublic URL: {public_url}')
    print(f'API Docs:   {public_url}/docs')
    print(f'GPU: {"CUDA" if torch.cuda.is_available() else "CPU"}')
    print('\nHow to use:')
    print('  1. Click the Public URL above')
    print('  2. Upload a traffic video (MP4/AVI/MOV)')
    print('  3. Click Auto-Label for detection')
    print('  4. Use Edit/Analytics/Lanes/Safety modes')
    print('  5. Export in COCO/YOLO/VOC/CSV format')
    print('\nKeep this cell running!')
    print('=' * 60)


In [ ]:
# Keep server running - press STOP to shut down
try:
    while True:
        time.sleep(60)
        if server.poll() is not None:
            print('Server stopped! Logs:')
            print(server.stderr.read().decode()[-3000:])
            break
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('Shut down.')


## (Optional) Upload a Video

In [ ]:
from google.colab import files
uploaded = files.upload()
for fn in uploaded:
    with open(f'/content/traffic-app/backend/uploads/{fn}', 'wb') as f:
        f.write(uploaded[fn])
    print(f'Saved: {fn} ({len(uploaded[fn])/1024/1024:.1f} MB)')


## (Optional) Server Status

In [ ]:
if server.poll() is None:
    print('Server is running')
else:
    print('Server stopped')
    print(server.stderr.read().decode()[-2000:])
